In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from pathlib import Path
import joblib
import shap

/home/mushman/Downloads/DCRM_analysis /.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
UNLABELLED_PATH = Path('..', '..', 'Dataset', 'unlabeled', '1.csv')

UNLABELLED_PATH.exists()


True

In [13]:
# Read raw lines first
lines = []
with open(UNLABELLED_PATH, 'r', encoding='latin-1') as f:
    for line in f:
        lines.append(line.strip().split(','))

# Now access metadata manually
for i, line in enumerate(lines[:44]):
    print(f"Row {i}: {line}")

Row 0: ['SAP ID', '109000226130', '']
Row 1: ['HEADER:', '']
Row 2: ['Substation:', 'INDORE', '', 'SAP Equipment No.:', '109000226130', '', 'Test Date :', '05-11-2020', '', '']
Row 3: ['Bay :', 'MAIN BAY BR-1', '', 'Breaker Type :', 'LTB800E4', '', 'Serial Number :', '1HSB01134067', '', '']
Row 4: ['Employee Id :', '60021298', '', 'Manufacturer :', 'ABB', '', 'Gas Pressure :', '7.0', '', '']
Row 5: ['Oil Pressure :', 'NIL', '', 'Air Pressure :', 'NIL', '', 'Voltage (kV) :', '800', '', '']
Row 6: ['']
Row 7: ['SETTINGS :', '']
Row 8: ['Trigger Option :', 'DCRM', '', 'Sampling Speed (kC) :', '10', '', 'Reistance Range uOhm :', '4000', '', '']
Row 9: ['Plot Length(ms) :', '500', '', 'Delay tco (ms) :', '300', '', 'Delay toc (ms) :', '300', '', '']
Row 10: ['Current Range C1(A) :', '25', '', 'Current Range C2(A) :', '25', '', 'Current Range C3(A) :', '25', '', '']
Row 11: ['Current Range C4(A) :', '25', '', 'Current Range C5(A) :', '25', '', 'Current Range C6(A) :', '25', '', '']
Row 12: [

In [24]:
def extract_all_metadata(lines, metadata_end=40):
    metadata = {}
    
    for line in lines[:metadata_end]:
        i = 0
        while i < len(line) - 1:
            key = str(line[i]).strip().rstrip(':').strip()
            val = str(line[i + 1]).strip()
            
            if key and key != 'nan' and val and val != 'nan':
                try:
                    float(key)
                    # Key is a number → swap them
                    metadata[val.rstrip(':').strip()] = key
                except ValueError:
                    # Normal key-value pair
                    metadata[key] = val
            i += 1
    
    return metadata

In [ ]:
metadata = extract_all_metadata(lines)
metadata

{'SAP ID': '109000226130',
 'Substation': 'INDORE',
 'SAP Equipment No.': '109000226130',
 'Test Date': '05-11-2020',
 'Bay': 'MAIN BAY BR-1',
 'Breaker Type': 'LTB800E4',
 'Serial Number': '1HSB01134067',
 'Employee Id': '60021298',
 'Manufacturer': 'ABB',
 'Gas Pressure': '7.0',
 'Oil Pressure': 'NIL',
 'Air Pressure': 'NIL',
 'Voltage (kV)': '800',
 'Trigger Option': 'DCRM',
 'Sampling Speed (kC)': '10',
 'Reistance Range uOhm': '4000',
 'Plot Length(ms)': '500',
 'Delay tco (ms)': '300',
 'Delay toc (ms)': '300',
 'Current Range C1(A)': '25',
 'Current Range C2(A)': '25',
 'Current Range C3(A)': '25',
 'Current Range C4(A)': '25',
 'Current Range C5(A)': '25',
 'Current Range C6(A)': '25',
 'T1 (mm)': '0.00',
 'T2 (mm)': '55.12',
 'T3 (mm)': '0.00',
 'T4 (mm)': '0.00',
 'T5 (mm)': '0.00',
 'T6 (mm)': '0.00',
 'Velocity': 'Disabled',
 'PIR': 'Disabled',
 'Auxiliary': 'Disabled',
 'Analog Channels': 'Disabled',
 'Linkage Ratio': '2.13',
 'Electric Stroke Length T1': '100',
 'Electric

In [29]:
df1 = pd.DataFrame.from_dict([metadata], orient='columns')
df1.head()

,SAP ID,Substation,SAP Equipment No.,Test Date,Bay,Breaker Type,Serial Number,Employee Id,Manufacturer,Gas Pressure,...,Open -Rebound(T3)(mm),Open - Rebound(T4)(mm),Open - Rebound(T5)(mm),Open - Rebound(T6)(mm),Coil Current (A)(C1),Coil Current (A)(C2),Coil Current (A)(C3),Coil Current (A)(C4),Coil Current (A)(C5),Coil Current (A)(C6)
0,109000226130,INDORE,109000226130,05-11-2020,MAIN BAY BR-1,LTB800E4,1HSB01134067,60021298,ABB,7.0,...,0.00,0.00,0.00,0.00,1.553,1.575,1.553,1.539,0.975,1.268
